# SVG Scaling Laws — End-to-End Runner

Run cells top to bottom on **Colab Pro** (A100 high-RAM) or **NYU Torch OOD** (H200).

**On NYU Torch:** run `scripts/setup_torch.sh` once from a login node first, then open this
notebook via https://ood.torch.hpc.nyu.edu and select the `Python (svg-scaling)` kernel.

After the LR sweep cells, look at the printed table, copy the best LR, and paste it into the next cell.

All numbers feed into `results/*.json`; the final cell builds the LaTeX report.

## 0. Environment

In [ ]:
import os
if os.path.exists('/content'):
    if not os.path.exists('/content/svg-scaling-laws'):
        !git clone https://github.com/ayushs2k1/svg-scaling-laws.git
    %cd /content/svg-scaling-laws
else:
    %cd ..
!nvidia-smi | head -20

In [ ]:
import os
# On Colab: install everything. On Torch OOD: conda env from setup_torch.sh already has all deps.
if os.path.exists('/content'):
    !pip install -q -r requirements.txt

## 1. Data: download, clean, tokenize, pack (~30 min, one-time)

In [ ]:
!python -m src.data.prepare \
    --out data/raw \
    --datasets starvector/svg-icons-simple starvector/svg-emoji-simple \
    --max-chars 8192 --min-chars 50

In [ ]:
!python -m src.tokenizer.train_bpe --in data/raw --out data/tok --vocab-size 4096

In [ ]:
!python -m src.data.pack --in data/raw --tok data/tok --out data/bin --seq-len 1024

In [ ]:
# Sequence-length histogram + a couple rendered examples
import json, numpy as np, matplotlib.pyplot as plt
stats = json.load(open('data/raw/stats.json'))
meta = json.load(open('data/bin/meta.json'))
print(json.dumps({**stats, **{'vocab_size': meta['vocab_size']}}, indent=2))
lens = np.load('data/bin/train_seq_lens.npy')
plt.figure(figsize=(6,3)); plt.hist(lens, bins=80); plt.xlabel('tokens / SVG'); plt.ylabel('count'); plt.title('Train sequence-length distribution'); plt.show()

import cairosvg, io, base64, random
from IPython.display import HTML, display
lines = open('data/raw/train.txt').readlines()
random.seed(0); picks = random.sample(lines, 6)
html = '<div style="display:flex;gap:8px;flex-wrap:wrap">'
for s in picks:
    try:
        png = cairosvg.svg2png(bytestring=s.strip().encode(), output_width=96)
        b64 = base64.b64encode(png).decode()
        html += f'<img src="data:image/png;base64,{b64}" style="background:#eee;padding:4px"/>'
    except Exception as e: html += f'<div>render fail: {e}</div>'
display(HTML(html + '</div>'))

## 2. LR sweep on Tiny — Standard Parameterization

In [ ]:
!python -m src.sweep \
    --config configs/tiny.yaml --data data/bin \
    --out-runs results/runs --out results/sweep_sp.json \
    --tag-prefix tiny_sp \
    --lrs 3e-5 1e-4 3e-4 1e-3 3e-3 1e-2 3e-2

In [ ]:
import json, matplotlib.pyplot as plt
s = json.load(open('results/sweep_sp.json'))
lrs = [r['lr'] for r in s['results']]; vs = [r['val_loss'] for r in s['results']]
plt.figure(figsize=(5,3)); plt.semilogx(lrs, vs, 'o-'); plt.xlabel('LR'); plt.ylabel('val loss'); plt.title('Tiny SP LR sweep'); plt.grid(True); plt.show()
print('Best SP LR:', s['best'])

## 3. Part 2: train all SP sizes at the best SP LR
**Set `BEST_LR_SP` from the cell above.**

In [ ]:
import json
BEST_LR_SP = json.load(open('results/sweep_sp.json'))['best']['lr']
print('Using BEST_LR_SP =', BEST_LR_SP)

In [ ]:
for size in ['tiny','small','medium','large','xl']:
    !python -m src.train --config configs/{size}.yaml --data data/bin \
        --out results/runs --tag sp_{size} --lr {BEST_LR_SP} --epochs 1

In [ ]:
# Training curves for all SP runs
!python -m src.plot_curves --runs results/runs --pattern "sp_*" --out results/sp_training_curves.png
from IPython.display import Image; Image('results/sp_training_curves.png')

## 4. LR sweep on Tiny — µP

In [ ]:
!python -m src.sweep --mup \
    --config configs/tiny.yaml --data data/bin \
    --out-runs results/runs --out results/sweep_mup.json \
    --tag-prefix tiny_mup \
    --lrs 3e-5 1e-4 3e-4 1e-3 3e-3 1e-2 3e-2

In [ ]:
import json, matplotlib.pyplot as plt
s_sp = json.load(open('results/sweep_sp.json'))
s_mu = json.load(open('results/sweep_mup.json'))
plt.figure(figsize=(5,3))
for s,lab in [(s_sp,'SP'),(s_mu,'µP')]:
    plt.semilogx([r['lr'] for r in s['results']], [r['val_loss'] for r in s['results']], 'o-', label=lab)
plt.xlabel('LR'); plt.ylabel('val loss'); plt.legend(); plt.grid(True); plt.title('Tiny LR sweep: SP vs µP'); plt.show()
print('Best µP LR:', s_mu['best'])

## 5. Part 3: train all µP sizes at the best µP LR

In [ ]:
import json
BEST_LR_MUP = json.load(open('results/sweep_mup.json'))['best']['lr']
print('Using BEST_LR_MUP =', BEST_LR_MUP)

In [ ]:
# Training curves for all µP runs
!python -m src.plot_curves --runs results/runs --pattern "mup_*" --out results/mup_training_curves.png
# All runs combined (SP + µP) — used in the report
!python -m src.plot_curves --runs results/runs --pattern "*" --out results/all_training_curves.png
from IPython.display import Image, display
display(Image('results/mup_training_curves.png'))
display(Image('results/all_training_curves.png'))

In [ ]:
for size in ['tiny','small','medium','large','xl']:
    !python -m src.train --config configs/{size}.yaml --data data/bin \
        --out results/runs --tag mup_{size} --lr {BEST_LR_MUP} --epochs 1 --mup --base-d-model 128

## 6. Scaling-law fit + extrapolation

In [ ]:
!python -m src.scaling_fit --runs results/runs --out results/scaling.json --plot results/scaling.png
from IPython.display import Image; Image('results/scaling.png')

## 7. Part 4: best model (XL µP), extra epochs, generate, evaluate

In [ ]:
BEST_EPOCHS = 3
!python -m src.train --config configs/xl.yaml --data data/bin \
    --out results/runs --tag best --lr {BEST_LR_MUP} --epochs {BEST_EPOCHS} --mup --base-d-model 128

In [ ]:
!python -m src.generate --ckpt results/runs/best/ckpt.pt --tok data/tok \
    --out results/samples --n-uncond 10 --temperatures 0.5 0.8 1.0 --top-p 0.95

In [ ]:
!python -m src.eval --ckpt results/runs/best/ckpt.pt --data data/bin \
    --samples results/samples --out results/eval.json

In [ ]:
# show generated PNG grid
import glob, base64
from IPython.display import HTML, display
html = '<div style="display:grid;grid-template-columns:repeat(5,96px);gap:6px">'
for p in sorted(glob.glob('results/samples/png/*.png'))[:25]:
    b64 = base64.b64encode(open(p,'rb').read()).decode()
    html += f'<img src="data:image/png;base64,{b64}" style="background:#eee;padding:2px"/>'
display(HTML(html + '</div>'))

## 8. Build the LaTeX report

In [ ]:
!python report/build_tables.py
# On Colab: install texlive-full or just download report/_auto and report/report.tex and compile locally.
# !apt-get -qq install texlive-latex-extra latexmk && cd report && latexmk -pdf report.tex